In [20]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START , END

from langgraph.graph.message import add_messages
## add message is a reducer for appending messages to the state graph

In [21]:
class State(TypedDict):

    messages: Annotated[list,add_messages]

graph_builder=StateGraph(State)

In [22]:
from dotenv import load_dotenv
load_dotenv()



True

In [23]:
from langchain_groq import ChatGroq

model=ChatGroq(
    model="openai/gpt-oss-20b"
)

model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C6FABAE790>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C6FABB3510>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### chatbot is like a node, and for every node we need a node definition 


In [24]:
## node functionality

def chatbot(state:State):
    return {"messages":[model.invoke(state["messages"])]}  ## state["messages"] this message is the user input, and the response will be appended in the messages variable(outer message in this code.)
    

In [25]:
## building graph

graph_builder= StateGraph(State)

## adding node

graph_builder.add_node("llmchatbot",chatbot) ## the name of the node, and the node definition

graph_builder.add_edge(START,"llmchatbot")

graph_builder.add_edge("llmchatbot",END)

# compile the graph
graph=graph_builder.compile()
 

In [26]:
response=graph.invoke({"messages":"Hi"})
response["messages"][-1].content

'Hello! How can I assist you today?'

### chatbot with tool

In [27]:

from langchain_tavily import TavilySearch

tool=TavilySearch(max_results=2)
tool.invoke("what is langsmith?")

{'query': 'what is langsmith?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.geeksforgeeks.org/nlp/introduction-to-langsmith',
   'title': 'Introduction to Langsmith - GeeksforGeeks',
   'content': "Langsmith is a framework designed to enhance and streamline the development of natural language processing (NLP) applications. It builds upon LangChain, a popular library for chaining multiple language models together, to create complex and flexible NLP workflows. Langsmith provides tools for managing, deploying, and scaling NLP applications efficiently.\n\n## What is Langsmith? [...] Langsmith is a comprehensive solution for developers and data scientists working with NLP models. It offers a robust environment for creating, testing and deploying language models in a maintainable manner. By leveraging LangChain'scapabilities, Langsmith simplifies the process of chaining multiple language models and other NLP components to build powerful app

In [28]:
def divide(a:int,b:int)->int:

    """
    divide a with b
    """
    return a/b
    

In [29]:
tools=[tool,divide]

llm_with_tools=model.bind_tools(tools)


In [30]:
llm_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C6FABAE790>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C6FABB3510>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'A search engine optimized for comprehensive, accurate, and 

In [31]:
from langgraph.graph import StateGraph,START,END,MessagesState

from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode

memory= MemorySaver()

#node definition

def tool_calling(state:MessagesState):

    return {"messages":[llm_with_tools.invoke(state["messages"])]}


## graph

builder =StateGraph(MessagesState)

builder.add_node("tool_calling_llm",tool_calling)

builder.add_node("tools",ToolNode(tools))

## add edges

builder.add_edge(START,"tool_calling_llm")

builder.add_conditional_edges(

    "tool_calling_llm",

    tools_condition
)

builder.add_edge("tools",END)

## compile the graph

graph=builder.compile(checkpointer=memory) 








### adding memory in langgraph

## creating a thread for a particular session


In [33]:
config={"configurable":{"thread_id":"1"}}

response=graph.invoke({"messages":"hi my name is dev"},config=config)
response['messages'][-1].content

'Hello Dev! 👋 How can I assist you today?'

In [34]:
response=graph.invoke({"messages":"what is my name?"},config=config)
response['messages'][-1].content

'You’re Dev!'